In [ ]:
from ssb import *

# Initialize the builder
builder = StrategicSegmentBuilder(
    target="target_cc_take",
    n_jobs=-1,
    min_sample_size=5000,
    min_lift=1.5,
    top_n_vars=30,
    max_segments=10,
    enable_diversity=False,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=False,  # Exclude 2-way rules from final output
    enable_3way=False,  # Exclude 3-way rules from final output
    feature_groups=None,
    ignore_features=["customer_id"]
)

data = pd.read_csv(r"synthetic_dataset_cc_takeup.csv")

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-06-29 03:15:46,151 | INFO     | [SSB.py:306] | Iteration 1 | Remaining Volume: 2,500,000 | Base Rate: 45.00%
2026-06-29 03:16:13,576 | INFO     | [SSB.py:387] | Segment 1 Captured: credit_score >= 787.50
2026-06-29 03:16:15,107 | INFO     | [SSB.py:306] | Iteration 2 | Remaining Volume: 2,346,323 | Base Rate: 42.89%
2026-06-29 03:16:47,394 | INFO     | [SSB.py:387] | Segment 2 Captured: avg_monthly_spend >= 21291.76
2026-06-29 03:16:48,507 | INFO     | [SSB.py:306] | Iteration 3 | Remaining Volume: 2,167,858 | Base Rate: 40.23%
2026-06-29 03:17:16,243 | INFO     | [SSB.py:387] | Segment 3 Captured: credit_score >= 752.50
2026-06-29 03:17:17,434 | INFO     | [SSB.py:306] | Iteration 4 | Remaining Volume: 1,966,378 | Base Rate: 37.68%
2026-06-29 03:17:42,323 | INFO     | [SSB.py:387] | Segment 4 Captured: annual_income >= 893523.47
2026-06-29 03:17:43,343 | INFO     | [SSB.py:306] | Iteration 5 | Remaining Volume: 1,867,031 | Base Rate: 36.46%
2026-06-29 03:18:06,839 | INFO     | [S

In [4]:
from prettytable import PrettyTable
table = PrettyTable()
table.field_names = list(final_eval.columns)
for _, row in final_eval.iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate    |        lift        |
+---------+-------------+---------------+--------------------+--------------------+--------------------+--------------------+
|   0.0   |  1237734.0  |    356453.0   | 28.798837229970253 |        45.0        |      49.50936      | 0.6399741606660057 |
|   1.0   |   153677.0  |    118710.0   |  77.2464324524815  |        45.0        |      6.14708       | 1.7165873878329223 |
|   2.0   |   178465.0  |    134257.0   | 75.22875633877791  |        45.0        |       7.1386       | 1.6717501408617315 |
|   3.0   |   201480.0  |    131142.0   | 65.08933889219773  |        45.0        |       8.0592       | 1.4464297531599495 |
|   4.0   |   99347.0   |    60245.0    | 60.64098563620441  |        45.0        |      3.97388       | 1.34757745858

In [5]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in segments_df.iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   credit_score=[787.50, inf)
SQL Filter: credit_score >= 787.50
--------------------------------------------------
Segment ID: 2
Raw Rule:   avg_monthly_spend=[21291.76, inf)
SQL Filter: avg_monthly_spend >= 21291.76
--------------------------------------------------
Segment ID: 3
Raw Rule:   credit_score=[752.50, inf)
SQL Filter: credit_score >= 752.50
--------------------------------------------------
Segment ID: 4
Raw Rule:   annual_income=[893523.47, inf)
SQL Filter: annual_income >= 893523.47
--------------------------------------------------
Segment ID: 5
Raw Rule:   credit_score=[738.50, inf)
SQL Filter: credit_score >= 738.50
--------------------------------------------------
Segment ID: 6
Raw Rule:   avg_monthly_spend=[17991.20, inf)
SQL Filter: avg_monthly_spend >= 17991.20
--------------------------------------------------
Segment ID: 7
Raw Rule:   credit_score=[720.50, inf)
SQL Filter: credit_score >= 720.50
--------------

In [6]:
final_eval

,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,1237734,356453.0,28.798837,45.0,49.50936,0.639974
1,1,153677,118710.0,77.246432,45.0,6.14708,1.716587
2,2,178465,134257.0,75.228756,45.0,7.13860,1.671750
3,3,201480,131142.0,65.089339,45.0,8.05920,1.446430
4,4,99347,60245.0,60.640986,45.0,3.97388,1.347577
5,5,112417,65107.0,57.915618,45.0,4.49668,1.287014
6,6,99127,54376.0,54.854883,45.0,3.96508,1.218997
7,7,163293,85583.0,52.410697,45.0,6.53172,1.164682
8,8,92944,45166.0,48.594853,45.0,3.71776,1.079886
9,9,87509,40950.0,46.795187,45.0,3.50036,1.039893


In [7]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
    SELECT *, 
    CASE WHEN credit_score >= 787.50 THEN 1 ELSE 0 END AS seg_1,
    CASE WHEN avg_monthly_spend >= 21291.76 THEN 1 ELSE 0 END AS seg_2,
    CASE WHEN credit_score >= 752.50 THEN 1 ELSE 0 END AS seg_3,
    CASE WHEN annual_income >= 893523.47 THEN 1 ELSE 0 END AS seg_4,
    CASE WHEN credit_score >= 738.50 THEN 1 ELSE 0 END AS seg_5,
    CASE WHEN avg_monthly_spend >= 17991.20 THEN 1 ELSE 0 END AS seg_6,
    CASE WHEN credit_score >= 720.50 THEN 1 ELSE 0 END AS seg_7,                  
    CASE WHEN credit_score >= 711.50 THEN 1 ELSE 0 END AS seg_8,       
    CASE WHEN credit_score >= 703.50 THEN 1 ELSE 0 END AS seg_9,    
    CASE WHEN annual_income >= 770299.44 THEN 1 ELSE 0 END AS seg_10,  

    CASE WHEN credit_score >= 787.50 THEN 1
    WHEN avg_monthly_spend >= 21291.76 THEN 2
    WHEN credit_score >= 752.50 THEN 3
    WHEN annual_income >= 893523.47 THEN 4
    WHEN credit_score >= 738.50 THEN 5
    WHEN avg_monthly_spend >= 17991.20 THEN 6
    WHEN credit_score >= 720.50 THEN 7
    WHEN credit_score >= 711.50 THEN 8
    WHEN credit_score >= 703.50 THEN 9
    WHEN annual_income >= 770299.44 THEN 10
    ELSE 0 END AS segment_id
    FROM predicted
""").df()
conn.close()

In [8]:
scorer = StrategicSegmentScore(
    target_col="target_cc_take",
    primary_key="customer_id",
    segment_cols=["seg_1", "seg_2", "seg_3", "seg_4", "seg_5", "seg_6", "seg_7", "seg_8", "seg_9", "seg_10"],
)

In [9]:
scorer.calculate_and_export_weights(predicted, export_path="scorecard_model_test1.json")

2026-06-29 03:26:35,367 | INFO     | [SSB.py:453] | Initializing DuckDB scorecard engine for 2,500,000 records...
2026-06-29 03:26:35,861 | INFO     | [SSB.py:487] | Computing harmonic scorecard weights...
2026-06-29 03:26:35,862 | INFO     | [SSB.py:524] | Scoring training dataset via NumPy Linear Algebra engine...
2026-06-29 03:26:35,899 | INFO     | [SSB.py:535] | Dataset Zero-Inflation Rate: 55.00%
2026-06-29 03:26:35,900 | INFO     | [SSB.py:541] | Normal Distribution (<80%). Deciling across full dataset...
2026-06-29 03:26:35,901 | INFO     | [SSB.py:550] | Calibrating deciles across 2,500,000 target customers...


{'model_metadata': {'total_training_population': 2500000,
  'active_scored_population': 2500000,
  'active_population_pct': 100.0,
  'baseline_event_rate': 0.45},
 'segment_weights': {'seg_1': {'weight': 32,
   'lift': 1.7166,
   'response_rate': 0.7725,
   'capture_rate': 0.1055},
  'seg_2': {'weight': 38,
   'lift': 1.6998,
   'response_rate': 0.7649,
   'capture_rate': 0.1294},
  'seg_3': {'weight': 56,
   'lift': 1.5841,
   'response_rate': 0.7129,
   'capture_rate': 0.2356},
  'seg_4': {'weight': 44,
   'lift': 1.6179,
   'response_rate': 0.7281,
   'capture_rate': 0.1686},
  'seg_5': {'weight': 65,
   'lift': 1.5275,
   'response_rate': 0.6874,
   'capture_rate': 0.3053},
  'seg_6': {'weight': 53,
   'lift': 1.5544,
   'response_rate': 0.6995,
   'capture_rate': 0.2248},
  'seg_7': {'weight': 73,
   'lift': 1.4558,
   'response_rate': 0.6551,
   'capture_rate': 0.4059},
  'seg_8': {'weight': 76,
   'lift': 1.4194,
   'response_rate': 0.6387,
   'capture_rate': 0.4595},
  'seg_9':

In [10]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/scorecard_model_test1.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-06-29 03:26:36,266 | INFO     | [2422015793.py:2] | Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/scorecard_model_test1.json...


In [11]:
model_artifact.get("decile_min_thresholds")

{'1': 348,
 '2': 292,
 '3': 193,
 '4': 111,
 '5': 53,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [12]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 32
Segment: seg_2 | Weight: 38
Segment: seg_3 | Weight: 56
Segment: seg_4 | Weight: 44
Segment: seg_5 | Weight: 65
Segment: seg_6 | Weight: 53
Segment: seg_7 | Weight: 73
Segment: seg_8 | Weight: 76
Segment: seg_9 | Weight: 78
Segment: seg_10 | Weight: 58


In [13]:
model_artifact.get("decile_min_thresholds")

{'1': 348,
 '2': 292,
 '3': 193,
 '4': 111,
 '5': 53,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [14]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 32 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 38 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 56 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 44 ELSE 0 END AS seg_4_weighted,
    CASE WHEN seg_5 = 1 THEN 65 ELSE 0 END AS seg_5_weighted,
    CASE WHEN seg_6 = 1 THEN 53 ELSE 0 END AS seg_6_weighted,
    CASE WHEN seg_7 = 1 THEN 73 ELSE 0 END AS seg_7_weighted,
    CASE WHEN seg_8 = 1 THEN 76 ELSE 0 END AS seg_8_weighted,
    CASE WHEN seg_9 = 1 THEN 78 ELSE 0 END AS seg_9_weighted,
    CASE WHEN seg_10 = 1 THEN 58 ELSE 0 END AS seg_10_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted + seg_5_weighted+ seg_6_weighted+ seg_7_weighted+ seg_8_weighted+ seg_9_weighted+ seg_10_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=348 THEN 1
                    WHEN total_weight >= 292 THEN 2
                    WHEN total_weight >= 193 THEN 3
                    WHEN total_weight >= 111 THEN 4
                    WHEN total_weight >= 53 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [15]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(target_cc_take) AS events, 
                    (SUM(target_cc_take)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()

In [16]:
scored

,decile_band,count,events,response_rate
0,1,414293,300638.0,72.566517
1,2,124262,75126.0,60.457743
2,3,284281,174001.0,61.207397
3,4,189390,100875.0,53.263108
4,5,250040,117907.0,47.155255
5,6,1237734,356453.0,28.798837
